# Health Retailers — Sales Anomaly Detection (cahier §11.2)
## Complete, executable Data Preparation Lab

**Objective:** transform raw sales exports into a traceable, model-ready dataset for **unsupervised detection of
unusual sales lines** (excessive discounts, sales below cost, atypical quantities or prices, unusual hours).

**Analytical grain:** **one row = one product line of a validated customer sale** (`documentItemm.Id`).

**Data used:** the **real export only** by default (`INCLUDE_SYNTHETIC = False`). Anomalies in the generated months
would only be simulated ones; the business must review real lines first.

**Expected business output:** a ranked **list of sales lines to check**. A flagged line is a candidate for review,
not a proven error or fraud.

**Team division:** Person 1 = sales sources, cleaning, cost reconstruction, line-level features. Person 2 = product,
seller and time context, feature selection, encoding, scaling and PCA. Both validate the final dataset.

# 1 — Business objective and definition of an anomaly

**Problem:** thousands of sales lines are recorded every month. A manager cannot read them all. Some deserve a
second look: a 60 % discount, a product sold below its cost, ten units of a product usually sold one at a time, a
sale at an unusual hour.

**Approach:** unsupervised anomaly detection, because **no line is labelled** "anomaly" in the data.
Business rules (discount > 50 %, sale below cost, atypical quantity) are **kept aside** to check the model's alerts;
they are not used as model inputs, otherwise the model would only learn the rules back.

**When is a line scored?** After the sale is validated (end-of-day review). Product and seller benchmarks are
computed on **earlier lines only**, so a line is compared with the history that existed when it was recorded.

**Full workflow**
1. **Data Understanding:** profile each source.
2. **Data Cleaning:** types, decimal commas, dates, drafts, duplicates, quality flags, quarantine.
3. **Data Transformation:** customer sales only, controlled joins to product, cost and seller.
4. **Feature Engineering:** discount, price and cost gaps, quantity versus product history, time, seller behaviour.
5. **Feature Selection:** relevance, redundancy, missingness, rules and identifiers kept out of the inputs.
6. **Dimensionality Reduction:** imputation, one-hot encoding, scaling, PCA.
7. **Optional anomaly-scoring extension:** Isolation Forest, review queue, comparison with the business rules.
8. **Validation and reporting.**

## How to study this notebook

Read each explanation **before** running the cell: it says **why** the step matters for sales anomalies, **what**
the commands do, **what output** to expect and **what to say** to the professor. Every number and chart comes from
executing the code on the project files.

## 0 — Setup and source files

| Source | Purpose | Key fields |
|---|---|---|
| `documentItemm.csv` | Product lines (the analytical grain) | `Id`, `Document_Id`, `ProductStock_Id`, `Quantity`, `PrixHt`, `DiscountPourcentage`, `PrixTotalHt` |
| `Documents.csv` | Header: date, time, type, state, seller | `Id`, `DocumentDate`, `CreatedDate`, `DocumentType_Id`, `State`, `CreatedBy` |
| `DocumentTypes.csv` | Which documents are sales | `Id`, `Name` |
| `RelatedDocuments.csv` | Invoices that regroup delivery notes | `Document1_Id`, `Document2_Id` |
| `ProductStocks.csv` | Stock line → product | `Id`, `Product_Id` |
| `StockMovements.csv` | Weighted average cost (CUMP) at the moment of each sale | `DocumentId`, `ProductStock_Id`, `Description`, `Extra` |
| `users_csv.csv` | Seller names | `Id`, `FullName` |
| `cleaned/products.csv` | Family and catalogue purchase price (key rebuilt by `script_cleaning_Produits.py`) | `Product_Id`, `FamilyGroup`, `Prix d'achat HT` |

### Guided code explanation — Setup

`INCLUDE_SYNTHETIC = False`: the lab works on the 7 real months. Switching it to `True` adds the generated months,
tagged by date, for a technical test of the pipeline only.

In [1]:
# STEP 0: Import the libraries and define every input/output path in one place.
# Raw exports are read from the project folders and are never modified.
# All generated evidence (profiles, audits, matrices, figures) goes to outputs/sales_anomaly/.
# Change PROJECT if the health-retailers repository is stored elsewhere on your computer.

# Run once if needed:
# %pip install pandas numpy matplotlib scikit-learn

from pathlib import Path
import io, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Locate the data branch checked out beside this branch, or set HEALTH_RETAILERS_DATA.
import os
_candidates = [Path(os.environ['HEALTH_RETAILERS_DATA']).expanduser()] if os.environ.get('HEALTH_RETAILERS_DATA') else []
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidates.extend([_base / 'data', _base.parent / 'data'])
PROJECT = next((_p.resolve() for _p in _candidates if (_p / 'original').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Check out the data branch beside ml-notebooks, or set HEALTH_RETAILERS_DATA to its root.')
RAW_DIR = PROJECT / 'original'                       # branch data-original: the 29 raw CSV files
SYNTH_DIR = PROJECT / 'generated'   # branch data-external: generated months
EXT_DIR = PROJECT / 'external'                        # weather and epidemics downloads
CLEAN_DIR = PROJECT / 'cleaned'                      # branch data-cleaned (product catalogue key)
OUTPUT_DIR = Path('outputs') / 'sales_anomaly'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REAL_START = pd.Timestamp('2024-09-30')
REAL_END = pd.Timestamp('2025-04-30 23:59:59')       # last day of the real export
SYNTH_END = pd.Timestamp('2026-09-29 23:59:59')      # last day of the generated data
INCLUDE_SYNTHETIC = False                  # True = add the generated months (tagged, never mixed silently)

for folder in [RAW_DIR, SYNTH_DIR, EXT_DIR, CLEAN_DIR]:
    print(f'{str(folder):60s}', 'found' if folder.exists() else 'MISSING')

data\original found
data\generated found
data\external found
data\cleaned found


### Guided code explanation — Reading and conversion helpers

**Why this step exists:** the 29 raw files were not exported consistently. Some are UTF-8, others Windows-1252
or DOS code page 850; some use `;`, others `,`; amounts use a decimal comma; dates mix `MM/DD/YYYY` and
`DD/MM/YYYY`. One set of helpers guarantees that every file follows the same rules.

- `read_text()` tries UTF-8 first, then keeps the legacy code page that produces real French letters
  (`pièce` instead of `pi�ce`).
- `load_csv()` detects `;` or `,` from the header line and reads **every column as text** (`dtype='string'`),
  so identifiers and amounts are never converted by accident before we inspect them.
- `parse_amount()` turns `42,38` into `42.38`; unreadable values become `NaN` and are counted.
- `parse_bool()` maps `VRAI/FAUX`, `TRUE/FALSE`, `1/0` to one Boolean type.
- `parse_date()` uses an **explicit format** per column. We never let pandas guess between day-first and
  month-first, because `03/04/2025` is ambiguous.
- `profile()` returns the Data Understanding evidence: type, missing values, distinct values.

**What to tell your professor:** "Every conversion is explicit and audited: a value that cannot be parsed
becomes missing and is counted, it is never invented."

In [2]:
# Helpers shared by every step of the notebook.
# Every column is read as text first; conversions are explicit and audited later.

def read_text(path):
    raw = path.read_bytes()
    try:
        return raw.decode('utf-8-sig')
    except UnicodeDecodeError:
        # Legacy export: keep the code page that yields real French letters.
        candidates = [raw.decode(cp, errors='replace') for cp in ('cp1252', 'cp850')]
        score = lambda t: sum(t.count(c) for c in 'éèàçêù°') - 3 * sum(t.count(c) for c in 'Š‚ø')
        return max(candidates, key=score)

def load_csv(path):
    text = read_text(path)
    header = text.split('\n', 1)[0]
    sep = ';' if header.count(';') > header.count(',') else ','
    df = pd.read_csv(io.StringIO(text), sep=sep, dtype='string')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def parse_amount(series):
    s = series.astype('string').str.strip().str.replace('\u00a0', '', regex=False).str.replace(' ', '', regex=False)
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce').astype('float64')

def parse_bool(series):
    mapping = {'vrai': True, 'true': True, '1': True, 'faux': False, 'false': False, '0': False}
    return series.astype('string').str.strip().str.lower().map(mapping).astype('boolean')

def parse_date(series, fmt):
    return pd.to_datetime(series.astype('string').str.strip(), format=fmt, errors='coerce')

def profile(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_%': (100 * df.isna().mean()).round(2),
        'unique_values': df.nunique(dropna=True),
    })

def safe_left_merge(left, right, left_on, right_on, label):
    """Left join that refuses to multiply rows: the lookup key must be unique."""
    right = right.dropna(subset=[right_on])
    if right[right_on].duplicated().any():
        raise ValueError(f'{label}: lookup key {right_on} is not unique, aggregate it first')
    n = len(left)
    out = left.merge(right, how='left', left_on=left_on, right_on=right_on,
                     suffixes=('', '_' + label), validate='many_to_one')
    assert len(out) == n, f'{label} multiplied rows!'
    matched = out[right_on].notna().mean() * 100
    print(f'{label:28s} {n:>7,} rows kept | {matched:5.1f} % matched')
    return out

conversion_log = []
def convert(df, name, amounts=(), bools=(), dates=None):
    """Apply the conversions and record how many non-empty values could not be parsed."""
    df = df.copy()
    for col in amounts:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_amount(df[col])
            conversion_log.append([name, col, 'amount', int(before - df[col].notna().sum())])
    for col in bools:
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_bool(df[col])
            conversion_log.append([name, col, 'boolean', int(before - df[col].notna().sum())])
    for col, fmt in (dates or {}).items():
        if col in df:
            before = df[col].notna().sum(); df[col] = parse_date(df[col], fmt)
            conversion_log.append([name, col, f'date {fmt}', int(before - df[col].notna().sum())])
    return df

print('Helpers ready.')

Helpers ready.


## 1 — Data Understanding: profile every dataset separately

### Guided code explanation — Profiling

- The overview table gives rows, columns, exact duplicates and fully empty columns for every source.
- The detailed profile of `documentItemm.csv` shows what one line contains and which columns are empty.
- `StockMovements.csv` is profiled because it hides the **cost** of each sale in a JSON field `Extra`
  (`{"prc": 23.62, "cump": 23.62}`); without it, "sold below cost" cannot be detected.
- `users_csv.csv` contains password hashes: they are **dropped immediately** and never displayed.

**Presentation use:** the overview table and the `documentItemm` profile.

In [3]:
# STEP 1: Read each dataset independently before any join.
SOURCES = {'documents': 'Documents.csv', 'items': 'documentItemm.csv', 'doc_types': 'DocumentTypes.csv',
           'related': 'RelatedDocuments.csv', 'product_stocks': 'ProductStocks.csv',
           'movements': 'StockMovements.csv', 'users': 'users_csv.csv'}
raw = {k: load_csv(RAW_DIR / f) for k, f in SOURCES.items()}
raw['users'] = raw['users'].drop(columns=['Password', 'PasswordSalt'], errors='ignore')   # secrets never used
synth = {k: load_csv(SYNTH_DIR / f) for k, f in SOURCES.items()
         if INCLUDE_SYNTHETIC and k in ('documents', 'items', 'movements') and (SYNTH_DIR / f).exists()}
catalogue = pd.read_csv(CLEAN_DIR / 'products.csv', dtype='string')

overview = pd.DataFrame([[SOURCES[k], len(df), df.shape[1], int(df.duplicated().sum()),
                          int((df.isna().mean() == 1).sum())] for k, df in raw.items()],
                        columns=['file', 'rows', 'columns', 'exact_duplicates', 'empty_columns'])
display(overview)
overview.to_csv(OUTPUT_DIR / 'sources_overview.csv', index=False)

for key in ['items', 'movements']:
    print(f'\n=== {SOURCES[key]}: {len(raw[key]):,} rows × {raw[key].shape[1]} columns ===')
    display(raw[key].head(3))
    p = profile(raw[key]); display(p)
    p.to_csv(OUTPUT_DIR / f'profile_{key}.csv', index_label='column')
display(raw['users'])

,file,rows,columns,exact_duplicates,empty_columns
0,Documents.csv,12542,35,0,2
1,documentItemm.csv,38705,34,0,10
2,DocumentTypes.csv,15,16,0,0
3,RelatedDocuments.csv,864,4,0,0
4,ProductStocks.csv,6842,5,0,0
5,StockMovements.csv,49609,11,0,2
6,users_csv.csv,7,4,0,0



=== documentItemm.csv: 38,705 rows × 34 columns ===


,Id,DisplayOrder,Description,Quantity,PrixHt,Tva,PrixTotalHt,PrixTotalTtc,DiscountPourcentage,DiscountAmount,Document_Id,ProductStock_Id,ProductPackage_Id,TotalTva,IncludedTaxes,TotalTaxes,SumPrixNetHt,SumQuantityUnit,SumDiscountTotal,SumPrixTotalHt,SumTotalTva,SumIncludedTaxes,SumTotalTaxes,SumPrixTotalTtc,PrixTtc,ForcedPrixTtc,SumPrixNetTtc,Extra,PrixTvc,ProductStockLocation_Id,BenTaxCategory_Id,NerTaxCategory_Id,TtcDiscountPercentage,TtcDiscountAmount
0,1,2,"<p style=""margin-top: 0px;margin-bottom: 0px;l...",1,"34,25",19,"27,4","32,61",20,<NA>,1,4157,<NA>,"5,21",<NA>,<NA>,"27,4",1,"6,85","27,4","5,206",0,0,"32,606","32,61",<NA>,"32,606",{},<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,2,"<p style=""margin-top: 0px;margin-bottom: 0px;l...",1,"44,51",19,"35,61","42,38",20,<NA>,2,387,<NA>,"6,77",<NA>,<NA>,"35,608",1,"8,9","35,61","6,7659",0,0,"42,3759","42,38",<NA>,"42,3759",{},<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3,2,"<p style=""margin-top: 0px;margin-bottom: 0px;l...",1,"12,3",19,"9,84","11,71",20,<NA>,3,3324,<NA>,"1,87",<NA>,<NA>,"9,84",1,"2,46","9,84","1,8696",0,0,"11,7096","11,71",<NA>,"11,7096",{},<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.00,38705
DisplayOrder,string,0,0.00,911
Description,string,0,0.00,5573
Quantity,string,1035,2.67,61
PrixHt,string,1079,2.79,6385
Tva,string,2472,6.39,2
PrixTotalHt,string,1079,2.79,11416
PrixTotalTtc,string,1079,2.79,11651
DiscountPourcentage,string,15077,38.95,88
DiscountAmount,string,38705,100.00,0



=== StockMovements.csv: 49,609 rows × 11 columns ===


,Id,DocumentId,MovementDate,Quantity,CurrentBalance,Description,ProductStock_Id,User_Id,Extra,ProductStockLocation_Id,CurrentLocationBalance
0,1,0,30/09/2024 09:05,1,1,Nouveau produit (Stock Initial),1,1,"{""prc"":193.28,""cump"":193.28}",<NA>,<NA>
1,2,0,30/09/2024 09:05,0,0,Nouveau produit (Stock Initial),2,1,"{""prc"":46.99,""cump"":46.99}",<NA>,<NA>
2,3,0,30/09/2024 09:05,0,0,Nouveau produit (Stock Initial),3,1,"{""prc"":140.00,""cump"":140.00}",<NA>,<NA>


,dtype,missing_count,missing_%,unique_values
Id,string,0,0.0,49609
DocumentId,string,0,0.0,12582
MovementDate,string,0,0.0,14219
Quantity,string,0,0.0,131
CurrentBalance,string,0,0.0,270
Description,string,0,0.0,13259
ProductStock_Id,string,0,0.0,6842
User_Id,string,0,0.0,6
Extra,string,0,0.0,12305
ProductStockLocation_Id,string,49609,100.0,0


,Id,FullName,UserName,Tresorerie
0,1,Administrateur,admin,"{""defaultCaisseId"":""5f5022c6-5883-45d4-bc76-d2..."
1,2,Système,_!_!_!_system_!_!_!_,<NA>
2,3,Marie,Marie,"{""defaultBankAccountId"":""41e8adb8-c852-40de-9f..."
3,4,Hugo,Hugo,"{""defaultBankAccountId"":""dede29c6-7d96-40fc-87..."
4,5,Celine,Celine,"{""defaultBankAccountId"":""c3800085-785e-4756-8d..."
5,6,melanie,melanie,"{""defaultBankAccountId"":""239b395c-ecdd-4cdb-80..."
6,7,kady,kady,"{""defaultCaisseId"":""86219038-b0d4-4b27-8e46-8b..."


## 2 — Data Cleaning: standardize without destroying anomalies

Rules:
- Decimal comma → decimal point; explicit date formats (`Documents`: month/day/year, `StockMovements`:
  day/month/year).
- **Do not delete** zero or negative quantities, zero prices or 100 % discounts: they may be the anomalies.
- Drafts (`State = 0`) are not sales; invoices that regroup delivery notes would count the same sale twice.
- Lines without a document or comment lines (no product, no amount) go to **quarantine**.

### Guided code explanation — Conversions, flags and quarantine

- `convert()` parses amounts, dates and Booleans and records unparseable values in `conversion_audit`.
- `SALE_TYPES = {1, 5, 6}`: customer invoice, delivery note, counter sale. Returns (`4`, `7`) are excluded from this
  dataset: a return is reviewed with its own logic (it is negative by nature).
- Each `flag_` column is a **visible** indicator. Only structural problems are removed (quarantine); business
  exclusions (drafts, regrouped invoices) are counted.

**What to tell the professor:** "Unusual values were preserved: removing them would hide the anomalies we want to
detect."

In [4]:
# STEP 2: Convert types and flag every quality issue.
def stack(key):
    return pd.concat([raw[key]] + ([synth[key]] if key in synth else []), ignore_index=True)

docs = convert(stack('documents'), 'documents', dates={'DocumentDate': '%m/%d/%Y %H:%M', 'CreatedDate': '%m/%d/%Y %H:%M'})
items = convert(stack('items'), 'items', amounts=['Quantity', 'PrixHt', 'PrixTotalHt', 'DiscountPourcentage'])
moves = convert(stack('movements'), 'movements', amounts=['Quantity'], dates={'MovementDate': '%d/%m/%Y %H:%M'})
for name, df in (('documents', docs), ('items', items), ('movements', moves)):
    print(f'{name}: {len(df):,} rows, exact duplicates = {int(df.duplicated().sum())}')
docs, items, moves = docs.drop_duplicates(), items.drop_duplicates(), moves.drop_duplicates()
assert docs['Id'].is_unique and items['Id'].is_unique

SALE_TYPES = {'1', '5', '6'}
grouped_invoices = set(raw['related']['Document1_Id']) & set(docs.loc[docs.DocumentType_Id == '1', 'Id'])
lines = items.merge(docs[['Id', 'DocumentDate', 'CreatedDate', 'DocumentType_Id', 'State', 'CreatedBy']],
                    how='left', left_on='Document_Id', right_on='Id', suffixes=('', '_doc'), validate='many_to_one')
assert len(lines) == len(items)
lines['flag_no_document'] = lines['Id_doc'].isna()
lines['flag_comment_line'] = lines['ProductStock_Id'].isna() & lines['PrixTotalHt'].fillna(0).eq(0)
lines['flag_draft'] = lines['State'].ne('1')
lines['flag_grouped_invoice'] = lines['Document_Id'].isin(grouped_invoices)
lines['flag_qty_nonpositive'] = lines['Quantity'].le(0)
lines['flag_price_zero'] = lines['PrixHt'].le(0)
lines['flag_discount_over_100'] = lines['DiscountPourcentage'].gt(100)
lines['flag_missing_time'] = lines['CreatedDate'].isna()
is_sale = lines['DocumentType_Id'].isin(SALE_TYPES)

flags = [c for c in lines if c.startswith('flag_')]
display(lines.loc[is_sale, flags].sum().to_frame('flagged_sales_lines'))
quarantine = lines[lines['flag_no_document'] | lines['flag_comment_line']]
quarantine.to_csv(OUTPUT_DIR / 'lines_quarantine.csv', index=False)
base = lines[is_sale & ~lines['flag_no_document'] & ~lines['flag_comment_line'] & ~lines['flag_draft']
             & ~lines['flag_grouped_invoice']].copy()
print(f'Sales lines retained: {len(base):,} | quarantined: {len(quarantine):,}')
conversion_audit = pd.DataFrame(conversion_log, columns=['dataset', 'column', 'conversion', 'unparseable_non_null'])
display(conversion_audit)
conversion_audit.to_csv(OUTPUT_DIR / 'conversion_audit.csv', index=False)

documents: 12,542 rows, exact duplicates = 0


items: 38,705 rows, exact duplicates = 0
movements: 49,609 rows, exact duplicates = 0


,flagged_sales_lines
flag_no_document,0
flag_comment_line,1024
flag_draft,925
flag_grouped_invoice,5638
flag_qty_nonpositive,117
flag_price_zero,2
flag_discount_over_100,1
flag_missing_time,0


Sales lines retained: 25,700 | quarantined: 1,038


,dataset,column,conversion,unparseable_non_null
0,documents,DocumentDate,date %m/%d/%Y %H:%M,0
1,documents,CreatedDate,date %m/%d/%Y %H:%M,0
2,items,Quantity,amount,0
3,items,PrixHt,amount,0
4,items,PrixTotalHt,amount,0
5,items,DiscountPourcentage,amount,0
6,movements,Quantity,amount,0
7,movements,MovementDate,date %d/%m/%Y %H:%M,0


## 3 — Controlled Data Transformation and Joins

1. `ProductStock_Id → ProductStocks.Id → Product_Id`, then `Product_Id → family, catalogue purchase price`.
2. **Cost of the line:** the stock movement written when the sale was validated carries the weighted average cost
   (`cump`). Key = (`DocumentId`, `ProductStock_Id`); several movements can exist (corrections), so the table is
   **aggregated to one cost per key first**. If no movement cost exists, the catalogue purchase price is used and the
   origin is recorded in `cost_source`.
3. `CreatedBy → users.Id` gives the seller name.

### Guided code explanation — Safe joins

- `safe_left_merge()` stops if a lookup key is duplicated, and asserts the number of lines did not change: a join can
  never multiply a sales line.
- `moves.Description.str.startswith('Validation')` keeps the movements created at validation (not later corrections).
- `json.loads` reads the `Extra` field; `cump` is extracted as a number.
- `cost_source` takes three values: `CUMP_MOVEMENT`, `CATALOGUE_PRICE`, `MISSING`. Missing costs are **not imputed**:
  a line without cost simply cannot be judged "below cost".

**Expected result:** same number of lines as `base`, unique line ids, share of lines with a cost.

In [5]:
# STEP 3: Enrich each sales line with product, cost and seller, never multiplying lines.
analytical = base.copy()
stock_map = raw['product_stocks'][['Id', 'Product_Id']].rename(columns={'Id': 'stock_id', 'Product_Id': 'product_id'})
analytical = safe_left_merge(analytical, stock_map, 'ProductStock_Id', 'stock_id', 'stock line -> product')

cat = catalogue.dropna(subset=['Product_Id']).drop_duplicates('Product_Id')
cat = cat[['Product_Id', 'FamilyGroup', "Prix d'achat HT"]].rename(
    columns={'Product_Id': 'cat_id', 'FamilyGroup': 'family', "Prix d'achat HT": 'catalogue_cost'})
cat['catalogue_cost'] = pd.to_numeric(cat['catalogue_cost'], errors='coerce').astype('float64')
analytical = safe_left_merge(analytical, cat, 'product_id', 'cat_id', 'product -> family, cost')

val = moves[moves['Description'].str.startswith('Validation', na=False) & moves['DocumentId'].ne('0')].copy()
val['cump'] = val['Extra'].map(lambda s: json.loads(s).get('cump') if isinstance(s, str) and s.startswith('{') else None)
val['cump'] = pd.to_numeric(val['cump'], errors='coerce')
val['cost_key'] = val['DocumentId'] + '|' + val['ProductStock_Id']
cost = val.dropna(subset=['cump']).sort_values('Id').groupby('cost_key', as_index=False)['cump'].first()
analytical['cost_key'] = analytical['Document_Id'] + '|' + analytical['ProductStock_Id']
analytical = safe_left_merge(analytical, cost, 'cost_key', 'cost_key', 'line -> CUMP cost')

analytical['unit_cost'] = analytical['cump'].fillna(analytical['catalogue_cost'].where(analytical['catalogue_cost'] > 0))
analytical['cost_source'] = np.select(
    [analytical['cump'].notna(), analytical['unit_cost'].notna()], ['CUMP_MOVEMENT', 'CATALOGUE_PRICE'], 'MISSING')
display(analytical['cost_source'].value_counts().to_frame('lines'))

users = raw['users'][['Id', 'FullName']].rename(columns={'Id': 'user_id', 'FullName': 'seller'})
analytical = safe_left_merge(analytical, users, 'CreatedBy', 'user_id', 'seller name')
assert analytical['Id'].is_unique and len(analytical) == len(base)

stock line -> product         25,700 rows kept | 100.0 % matched
product -> family, cost       25,700 rows kept |  91.8 % matched


line -> CUMP cost             25,700 rows kept | 100.0 % matched


,lines
cost_source,
CUMP_MOVEMENT,25640
CATALOGUE_PRICE,58
MISSING,2


seller name                   25,700 rows kept | 100.0 % matched


## 4 — Feature Engineering

| Signal | Feature | Why it can reveal an anomaly |
|---|---|---|
| Discount | `discount_pct` | very high or unusual discount |
| Price | `price_gap_vs_product_pct` | unit price far from the product's usual price |
| Margin | `margin_gap_pct` | line sold below (negative) or far above its cost |
| Quantity | `qty_z_vs_product` | quantity far from what this product usually sells |
| Basket | `lines_in_document` | unusually large document |
| Seller | `seller_past_avg_discount` | seller who usually discounts more |
| Time | `hour`, `weekday`, `is_saturday` | sale outside usual opening rhythm |
| Context | `channel`, `seller`, `family` | categorical context, one-hot encoded later |

### Guided code explanation — Feature engineering

- Lines are sorted by creation time; `groupby(...).shift(1)` + `expanding()` compute a benchmark **from earlier lines
  only**. Example: the "usual price" of a product is the median of its previous sales, not including the current
  one.
- `qty_z_vs_product = (quantity − past mean) / past standard deviation`, computed after at least 5 previous sales of
  the product; before that, it stays missing (treated by imputation in Step 6).
- `margin_gap_pct = (line total excl. VAT − quantity × unit cost) / (quantity × unit cost) × 100`: negative = sold
  below cost.
- `hour` comes from `CreatedDate` (`DocumentDate` has no time).
- The three **business rules** are computed here but stored as `rule_` columns for **audit only**.

**Leakage note:** no feature uses a later line. The rules are not inputs.

In [6]:
# STEP 4: Line-level signals, product and seller benchmarks computed on earlier lines only.
f = analytical.sort_values(['CreatedDate', 'Id']).reset_index(drop=True)
f['discount_pct'] = f['DiscountPourcentage'].fillna(0)
f['unit_price_ht'] = f['PrixHt']
f['quantity'] = f['Quantity']
f['line_total_ht'] = f['PrixTotalHt']
line_cost = f['quantity'] * f['unit_cost']
f['margin_gap_pct'] = np.where(line_cost > 0, 100 * (f['line_total_ht'] - line_cost) / line_cost.where(line_cost > 0), np.nan)

by_product = f.groupby('product_id', dropna=False)
past_price = by_product['unit_price_ht'].transform(lambda s: s.shift(1).expanding(min_periods=3).median())
f['price_gap_vs_product_pct'] = 100 * (f['unit_price_ht'] - past_price) / past_price.where(past_price > 0)
past_mean = by_product['quantity'].transform(lambda s: s.shift(1).expanding(min_periods=5).mean())
past_std = by_product['quantity'].transform(lambda s: s.shift(1).expanding(min_periods=5).std())
f['qty_z_vs_product'] = (f['quantity'] - past_mean) / past_std.where(past_std > 0)
f['lines_in_document'] = f.groupby('Document_Id')['Id'].transform('count')
f['seller_past_avg_discount'] = f.groupby('CreatedBy')['discount_pct'].transform(
    lambda s: s.shift(1).expanding(min_periods=20).mean())
f['hour'] = f['CreatedDate'].dt.hour
f['weekday'] = f['CreatedDate'].dt.weekday
f['is_saturday'] = f['weekday'].eq(5).astype('int8')
f['channel'] = f['DocumentType_Id'].map({'6': 'counter', '5': 'delivery_note', '1': 'invoice'})
f['seller'] = f['seller'].fillna('NON_RENSEIGNE')
f['family'] = f['family'].fillna('NON_RENSEIGNE')

# Business rules: for auditing the model's alerts, never as inputs.
f['rule_discount_over_50'] = f['discount_pct'].gt(50)
f['rule_below_cost'] = f['margin_gap_pct'].lt(0) & f['quantity'].gt(0)
f['rule_atypical_qty'] = f['qty_z_vs_product'].abs().gt(3)
display(f[['rule_discount_over_50', 'rule_below_cost', 'rule_atypical_qty']].sum().to_frame('lines'))
display(f[['Id', 'DocumentDate', 'seller', 'channel', 'quantity', 'unit_price_ht', 'discount_pct',
           'margin_gap_pct', 'price_gap_vs_product_pct', 'qty_z_vs_product']].head(8))

,lines
rule_discount_over_50,47
rule_below_cost,587
rule_atypical_qty,342


,Id,DocumentDate,seller,channel,quantity,unit_price_ht,discount_pct,margin_gap_pct,price_gap_vs_product_pct,qty_z_vs_product
0,2,2024-09-30,Marie,counter,1.0,44.51,20.0,15.993485,NaN,NaN
1,3,2024-09-30,Marie,counter,1.0,12.30,20.0,16.037736,NaN,NaN
2,4,2024-09-30,Marie,counter,1.0,3.70,0.0,40.151515,NaN,NaN
3,5,2024-09-30,Marie,counter,1.0,133.61,20.0,16.134289,NaN,NaN
4,6,2024-09-30,Marie,counter,1.0,15.46,0.0,44.892221,NaN,NaN
5,7,2024-09-30,Marie,counter,1.0,68.91,20.0,16.627882,NaN,NaN
6,8,2024-09-30,Marie,counter,1.0,107.56,20.0,11.898570,NaN,NaN
7,10,2024-09-30,Marie,counter,1.0,26.30,20.0,16.114790,NaN,NaN


## 5 — Feature Selection and Leakage Audit

### Guided code explanation — Feature selection

- `numeric_candidates` / `categorical_candidates` list the possible model inputs.
- `missingness` shows which inputs need imputation (benchmarks are missing for a product's first sales).
- The correlation heatmap reveals redundancy: for example `line_total_ht` grows with `quantity` and
  `unit_price_ht`; keeping all three would count the same information several times.
- Rule: if two numeric features have |r| > 0.85, the one less directly linked to an anomaly signal is dropped.
  `line_total_ht` is dropped in any case: it is an exact function of quantity, unit price and discount, even when the
  linear correlation looks moderate. Every decision is written in `selection`.
- `audit_only`: identifiers, raw dates and the three `rule_` columns. They stay in the review export, never in the
  model.

**What to say:** "Inputs were chosen for relevance, availability at scoring time, missingness and redundancy;
business rules were kept out to validate the model independently."

In [7]:
# STEP 5: Choose the inputs, document every decision.
numeric_candidates = ['discount_pct', 'unit_price_ht', 'quantity', 'line_total_ht', 'margin_gap_pct',
                      'price_gap_vs_product_pct', 'qty_z_vs_product', 'lines_in_document',
                      'seller_past_avg_discount', 'hour', 'weekday', 'is_saturday']
categorical_candidates = ['channel', 'seller', 'family']
audit_only = ['Id', 'Document_Id', 'product_id', 'DocumentDate', 'CreatedDate', 'cost_source',
              'rule_discount_over_50', 'rule_below_cost', 'rule_atypical_qty']

missingness = f[numeric_candidates + categorical_candidates].isna().mean().mul(100).round(2)
display(missingness.sort_values(ascending=False).to_frame('missing_%'))
corr = f[numeric_candidates].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(corr)), corr.columns, rotation=90); ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im, ax=ax); ax.set_title('Numeric candidates: correlation')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_correlation.png', dpi=160); plt.show()

pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(numeric_candidates)
         for b in numeric_candidates[i + 1:] if abs(corr.loc[a, b]) > 0.85]
print('Highly correlated pairs (|r| > 0.85):', pairs)
redundant = {'line_total_ht'}          # = quantity × price × (1 − discount): rebuilt from kept inputs
constant = [c for c in numeric_candidates + categorical_candidates if f[c].nunique(dropna=True) <= 1]
num_cols = [c for c in numeric_candidates if c not in redundant and c not in constant]
cat_cols = [c for c in categorical_candidates if c not in constant]
selection = pd.DataFrame({'feature': numeric_candidates + categorical_candidates + audit_only,
                          'role': ['numeric'] * len(numeric_candidates) + ['categorical'] * len(categorical_candidates)
                                  + ['audit_only'] * len(audit_only)})
selection['selected'] = selection['feature'].isin(num_cols + cat_cols)
selection['reason_if_dropped'] = np.select(
    [selection.feature.isin(redundant), selection.feature.isin(constant), selection.role.eq('audit_only')],
    ['redundant with quantity × price × discount', 'constant', 'traceability / validation only'], '')
display(selection)
selection.to_csv(OUTPUT_DIR / 'feature_selection.csv', index=False)

,missing_%
qty_z_vs_product,61.39
price_gap_vs_product_pct,33.90
margin_gap_pct,1.33
seller_past_avg_discount,0.47
unit_price_ht,0.03
line_total_ht,0.03
quantity,0.00
discount_pct,0.00
lines_in_document,0.00
hour,0.00


Highly correlated pairs (|r| > 0.85): []


C:\Users\firas\AppData\Local\Temp\ipykernel_20584\382545837.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_correlation.png', dpi=160); plt.show()


,feature,role,selected,reason_if_dropped
0,discount_pct,numeric,True,
1,unit_price_ht,numeric,True,
2,quantity,numeric,True,
3,line_total_ht,numeric,False,redundant with quantity × price × discount
4,margin_gap_pct,numeric,True,
5,price_gap_vs_product_pct,numeric,True,
6,qty_z_vs_product,numeric,True,
7,lines_in_document,numeric,True,
8,seller_past_avg_discount,numeric,True,
9,hour,numeric,True,


## 6 — Imputation, One-Hot Encoding, Standardization and PCA

1. Missing numeric values (benchmarks of a product's first sales, lines without cost) → **training median**.
2. Categories (`channel`, `seller`, `family`) → **one-hot encoding**.
3. **StandardScaler**: a discount in % and a quantity in units must weigh comparably in a distance.
4. **PCA** keeping at least 90 % of the variance, compared later with the full feature set.
5. Everything is fitted on the **earlier 80 %** of lines (chronological split), then applied to the later 20 %.

### Guided code explanation — Preprocessing and PCA

- `ColumnTransformer` = numeric pipeline (median imputation + scaling) and categorical pipeline (constant
  `NON_RENSEIGNE` + one-hot) side by side.
- `handle_unknown='ignore'`: a seller or family that appears only in the holdout does not crash the transformation.
- `PCA(n_components=0.90)` builds the smallest number of uncorrelated components keeping 90 % of the variance.

**Limitation:** PCA keeps variance, not necessarily rare anomalies; Step 10 compares both representations.

In [8]:
# STEP 6: Chronological split, then fit preprocessing on the training lines only.
ordered = f.sort_values('CreatedDate').reset_index(drop=True)
split = int(len(ordered) * 0.8)
train, holdout = ordered.iloc[:split].copy(), ordered.iloc[split:].copy()
print(f'Train: {len(train):,} lines up to {train.CreatedDate.max():%d/%m/%Y} | holdout: {len(holdout):,} lines')

num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='NON_RENSEIGNE')),
                         ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', num_pipeline, num_cols), ('cat', cat_pipeline, cat_cols)],
                                 remainder='drop', sparse_threshold=0)
X_train = preprocessor.fit_transform(train)
X_holdout = preprocessor.transform(holdout)
names = preprocessor.get_feature_names_out()
assert np.isfinite(X_train).all() and np.isfinite(X_holdout).all()
print('Feature matrix:', X_train.shape, '| holdout:', X_holdout.shape)

pca = PCA(n_components=0.90, svd_solver='full')
Z_train = pca.fit_transform(X_train)
Z_holdout = pca.transform(X_holdout)
print('PCA dimensions:', X_train.shape[1], '→', Z_train.shape[1],
      '| explained variance:', round(float(pca.explained_variance_ratio_.sum()), 4))

Train: 20,560 lines up to 24/03/2025 | holdout: 5,140 lines


Feature matrix: (20560, 27) | holdout: (5140, 27)


PCA dimensions: 27 → 10 | explained variance: 0.9141


## 7 — Visual Evidence for the Presentation

In [9]:
# STEP 7: Distribution of the main signals and PCA variance.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].hist(f['discount_pct'], bins=40); axes[0].set(title='Discount per line (%)', xlabel='%')
axes[1].hist(f['margin_gap_pct'].clip(-100, 300).dropna(), bins=60); axes[1].axvline(0, color='red')
axes[1].set(title='Margin gap vs cost (%) — left of red = below cost', xlabel='% (clipped)')
axes[2].hist(f['qty_z_vs_product'].clip(-5, 15).dropna(), bins=60); axes[2].axvline(3, color='red')
axes[2].set(title='Quantity z-score vs product history', xlabel='z (clipped)')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'signal_distributions.png', dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker='o')
ax.axhline(0.90, linestyle='--', label='90 % threshold')
ax.set(xlabel='components', ylabel='cumulative explained variance', title='PCA variance retained'); ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=160); plt.show()

C:\Users\firas\AppData\Local\Temp\ipykernel_20584\1158528007.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'signal_distributions.png', dpi=160); plt.show()


C:\Users\firas\AppData\Local\Temp\ipykernel_20584\1158528007.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=160); plt.show()


## 8 — Final Validation and Deliverables

In [10]:
# STEP 8: Checks and exports.
assert f['Id'].is_unique and len(f) == len(base)
assert not any(c in names for c in audit_only)
assert not any(n.startswith(('num__rule_', 'cat__rule_')) for n in names)
f.to_csv(OUTPUT_DIR / 'sales_lines_features.csv', index=False)
pd.DataFrame(X_train, columns=names).to_csv(OUTPUT_DIR / 'X_train_encoded.csv', index=False)
pd.DataFrame(X_holdout, columns=names).to_csv(OUTPUT_DIR / 'X_holdout_encoded.csv', index=False)
pd.DataFrame(Z_train).to_csv(OUTPUT_DIR / 'X_train_pca.csv', index=False)
pd.DataFrame(Z_holdout).to_csv(OUTPUT_DIR / 'X_holdout_pca.csv', index=False)
summary = pd.DataFrame([
    ['raw_lines', len(items)], ['sales_lines_retained', len(base)], ['quarantined_lines', len(quarantine)],
    ['lines_with_cost_%', round(100 * f['unit_cost'].notna().mean(), 2)],
    ['encoded_feature_count', X_train.shape[1]], ['pca_component_count', Z_train.shape[1]],
    ['pca_explained_variance', round(float(pca.explained_variance_ratio_.sum()), 4)],
], columns=['metric', 'value'], dtype=object)
display(summary)
summary.to_csv(OUTPUT_DIR / 'preparation_summary.csv', index=False)

,metric,value
0,raw_lines,38705
1,sales_lines_retained,25700
2,quarantined_lines,1038
3,lines_with_cost_%,99.99
4,encoded_feature_count,27
5,pca_component_count,10
6,pca_explained_variance,0.9141


# 9 — Optional extension: unsupervised anomaly scoring

**Isolation Forest** isolates observations with random splits; a line isolated quickly receives a lower score
(more unusual). `contamination = 0.02` is an **alert budget** (about 2 % of lines reviewed), not a measured anomaly
rate.

Because there are no labels, we use the business rules as an **independent check**: if the model is useful, the
share of rule-flagged lines among its alerts should be clearly higher than among all lines (a "lift").

In [11]:
# STEP 9: Fit on earlier lines, score later lines, compare alerts with the business rules.
from sklearn.ensemble import IsolationForest

model = IsolationForest(n_estimators=300, contamination=0.02, random_state=42, n_jobs=-1).fit(X_train)
holdout['anomaly_score'] = model.decision_function(X_holdout)
holdout['flagged_for_review'] = model.predict(X_holdout) == -1
holdout['any_rule'] = holdout[['rule_discount_over_50', 'rule_below_cost', 'rule_atypical_qty']].any(axis=1)

check = pd.DataFrame({
    'all holdout lines': holdout[['rule_discount_over_50', 'rule_below_cost', 'rule_atypical_qty', 'any_rule']].mean() * 100,
    'model alerts': holdout.loc[holdout.flagged_for_review,
                                ['rule_discount_over_50', 'rule_below_cost', 'rule_atypical_qty', 'any_rule']].mean() * 100,
}).round(2)
check['lift'] = (check['model alerts'] / check['all holdout lines']).round(1)
display(check)

review_cols = ['Id', 'Document_Id', 'DocumentDate', 'seller', 'channel', 'family', 'quantity', 'unit_price_ht',
               'discount_pct', 'margin_gap_pct', 'qty_z_vs_product', 'anomaly_score', 'any_rule']
review_queue = holdout.loc[holdout.flagged_for_review, review_cols].sort_values('anomaly_score')
review_queue.to_csv(OUTPUT_DIR / 'sales_review_queue.csv', index=False)
print(f'Holdout lines: {len(holdout):,} | flagged for review: {len(review_queue):,}')
display(review_queue.head(15))

,all holdout lines,model alerts,lift
rule_discount_over_50,0.16,5.95,37.2
rule_below_cost,0.93,9.52,10.2
rule_atypical_qty,1.40,8.33,6.0
any_rule,2.30,17.86,7.8


Holdout lines: 5,140 | flagged for review: 84


,Id,Document_Id,DocumentDate,seller,channel,family,quantity,unit_price_ht,discount_pct,margin_gap_pct,qty_z_vs_product,anomaly_score,any_rule
21534,41230,15690,2025-04-02,Celine,invoice,COMPLEMENTS ALIMENTAIRES,1.0,21.300,0.0,13.636364,-0.750939,-0.072347,False
21531,41228,15690,2025-04-02,Celine,invoice,COMPLEMENTS ALIMENTAIRES,1.0,4.950,0.0,-91.651910,NaN,-0.066066,True
21219,40104,15545,2025-03-28,Celine,invoice,TOILETTE,1.0,4.720,0.0,13.762352,-0.288675,-0.055310,False
23430,44566,17599,2025-04-16,Celine,invoice,NON_RENSEIGNE,1.0,26.186,0.0,15.739227,-0.436242,-0.053610,False
24525,45940,18097,2025-04-23,Celine,invoice,NON_RENSEIGNE,1.0,5.000,0.0,5.263158,-0.378785,-0.053534,False
24946,46520,18321,2025-04-25,Celine,invoice,TOILETTE,1.0,45.530,0.0,0.000000,NaN,-0.050843,False
23433,44569,17600,2025-04-16,Celine,invoice,TOILETTE,1.0,198.319,0.0,50.811781,-0.397984,-0.049525,False
22828,42853,16331,2025-04-11,Celine,invoice,TOILETTE,1.0,30.000,0.0,11.111111,-0.204124,-0.042829,False
22827,42852,16331,2025-04-11,Celine,invoice,TOILETTE,1.0,20.000,0.0,11.111111,NaN,-0.041533,False
22826,42851,16331,2025-04-11,Celine,invoice,TOILETTE,1.0,25.000,0.0,11.111111,NaN,-0.041252,False


## 10 — Optional comparison: PCA vs original features

The same detector is trained on the PCA components. Agreement between the two alert lists is a **stability check**,
not proof of correctness.

In [12]:
# STEP 10: Same detector on PCA components.
pca_model = IsolationForest(n_estimators=300, contamination=0.02, random_state=42, n_jobs=-1).fit(Z_train)
holdout['flag_pca'] = pca_model.predict(Z_holdout) == -1
both = (holdout.flagged_for_review & holdout.flag_pca).sum()
print(f'Alerts full features: {int(holdout.flagged_for_review.sum())} | PCA: {int(holdout.flag_pca.sum())} | in both lists: {int(both)}')
print('Agreement on all holdout lines:', round(float((holdout.flagged_for_review == holdout.flag_pca).mean()), 4))
print('Share of PCA alerts matching a business rule:', round(100 * holdout.loc[holdout.flag_pca, 'any_rule'].mean(), 2), '%')

Alerts full features: 84 | PCA: 59 | in both lists: 10
Agreement on all holdout lines: 0.9761
Share of PCA alerts matching a business rule: 37.29 %


### Reading the result

- The **lift** of Step 9 is the key figure: how many times more often the model's alerts break a business rule than
  an average line. A lift well above 1 shows the model points at genuinely unusual lines, without having seen the
  rules.
- The two alert lists (full features vs PCA) overlap only partly: they are two different views of "unusual". The one
  whose alerts match the rules more often is the better candidate, but the final word belongs to a human review of a
  sample of alerts.

## 11 — What to say during the presentation

**Person 1:** "We kept validated customer sales only, excluded invoices that regroup delivery notes already counted,
and quarantined lines without product or document. We rebuilt the cost of each line from the stock movement written
at validation, and kept unusual values instead of deleting them."

**Person 2:** "We compared every line with the history of its product and its seller, using earlier lines only. We
removed a redundant amount, imputed missing benchmarks with training medians, one-hot encoded the channel, seller
and family, scaled the inputs and tested a PCA."

**Both:** "The output is a ranked list of lines to check. Business rules were kept out of the model and used to verify
that its alerts make sense."

### Limitations to disclose
- No labelled anomalies: precision cannot be measured without a human review of the alerts.
- 2 % is a review budget, not the true anomaly rate.
- Lines without a movement cost use the catalogue price; a few have no cost at all.
- A seller with few lines has an unstable benchmark during the first weeks.